# FinTech 545 Midterm Answer Sheet

**Name:** Erqi Huang  

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt

from risk_management import *

DATA = Path("midterm data")
N_SIMULATIONS = 100_000
RANDOM_SEED = 545

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

required_files = [
    "problem1.csv",
    "problem2.csv",
    "problem3.csv",
    "problem3_holdings.csv",
]
missing_files = [name for name in required_files if not (DATA / name).exists()]
assert not missing_files, f"Missing files: {missing_files}"

print("Setup complete — the FinTech 545 Python library is ready.")
print(f"Simulation settings: {N_SIMULATIONS:,} draws; random seed = {RANDOM_SEED}.")

Setup complete — the FinTech 545 Python library is ready.
Simulation settings: 100,000 draws; random seed = 545.


# Question 1 — Pairwise Correlation with Missing Data

### Question 1(a): Calculate the pairwise correlation matrix, which estimates each correlation from the rows on which that particular pair is jointly observed. The numbers of usable observations are 41, 41, 36, 41, and 42 for x1 through x5.

In [2]:
# INPUT
q1_data = pd.read_csv(DATA / "problem1.csv")
q1_labels = list(q1_data.columns)

# FUNCTION CALL: pairwise correlation with missing values.
q1_diagnostics = correlation_diagnostics(q1_data)
q1_pairwise_correlation = pd.DataFrame(
    q1_diagnostics["pairwise"], index=q1_labels, columns=q1_labels
)

display(q1_data.notna().sum().to_frame("Available observations"))
display(q1_pairwise_correlation)

,Available observations
x1,41
x2,41
x3,36
x4,41
x5,42


,x1,x2,x3,x4,x5
x1,1.000000,0.332836,0.418462,0.255285,0.732272
x2,0.332836,1.000000,0.251035,0.287709,0.787538
x3,0.418462,0.251035,1.000000,0.483580,0.684257
x4,0.255285,0.287709,0.483580,1.000000,0.782632
x5,0.732272,0.787538,0.684257,0.782632,1.000000


The pairwise correlation matrix is

$$
\begin{bmatrix}
1.000000 & 0.332836 & 0.418462 & 0.255285 & 0.732272\\
0.332836 & 1.000000 & 0.251035 & 0.287709 & 0.787538\\
0.418462 & 0.251035 & 1.000000 & 0.483580 & 0.684257\\
0.255285 & 0.287709 & 0.483580 & 1.000000 & 0.782632\\
0.732272 & 0.787538 & 0.684257 & 0.782632 & 1.000000
\end{bmatrix}.
$$

### Question 1(b):Calculate the variance of each series using every available observation for that series.
With the absence of missing values, we can use all nonmissing values in each column. By squaring the sample std to gain sample variances.

In [3]:
# FUNCTION CALL: marginal sample standard deviations using all available data.
q1_standard_deviations = sample_standard_deviations(
    q1_data.to_numpy(dtype=float), skip_missing=True
)
q1_variances = q1_standard_deviations**2
q1_variance_table = pd.DataFrame(
    {
        "Available Observations": q1_data.notna().sum(),
        "Sample Variance": q1_variances,
    },
    index=q1_labels,
)
display(q1_variance_table)

,Available Observations,Sample Variance
x1,41,0.000406
x2,41,0.000480
x3,36,0.000151
x4,41,0.000842
x5,42,0.000273


The sample variances are:

| Series | Available observations | Variance |
|---|---:|---:|
| x1 | 41 | **0.000406074** |
| x2 | 41 | **0.000480066** |
| x3 | 36 | **0.000151110** |
| x4 | 41 | **0.000841739** |
| x5 | 42 | **0.000273150** |

### Question 1(c): Determine whether the pairwise correlation matrix is positive semidefinite and show the evidence.
Calculate all the eigenvalues for the matrix. A symmetric matrix is PSD only if every eigenvalue is nonnegative.

In [4]:
q1_original_eigenvalues = np.linalg.eigvalsh(
    q1_pairwise_correlation.to_numpy(dtype=float)
)
display(pd.DataFrame({"Eigenvalue": q1_original_eigenvalues}))
print("Minimum eigenvalue:", q1_original_eigenvalues.min())

Minimum eigenvalue: -0.08927866769735246


,Eigenvalue
0,-0.089279
1,0.461340
2,0.719085
3,0.823437
4,3.085416


The eigenvalues are -0.089279, 0.461340, 0.719085, 0.823437, and 3.085416. Because the minimum eigenvalue is negative, the pairwise correlation matrix is not PSD. 

### Question 1(d): Use Higham's method to obtain the nearest positive-semidefinite correlation matrix, report its eigenvalues, and identify which correlations changed the most.

Calulate its Higham result. And then rank unique off-diagonal changes by absolute size.

In [5]:
# FUNCTION CALL: Higham nearest-PSD correlation repair.
q1_repairs = repair_correlation_summary(
    q1_pairwise_correlation.to_numpy(dtype=float)
)
q1_higham_correlation = pd.DataFrame(
    q1_repairs["Higham"]["matrix"], index=q1_labels, columns=q1_labels
)
q1_higham_eigenvalues = np.linalg.eigvalsh(
    q1_higham_correlation.to_numpy(dtype=float)
)

# FUNCTION CALL: correlations with the largest repair movements.
q1_largest_movements = largest_matrix_movements(
    q1_pairwise_correlation.to_numpy(dtype=float),
    q1_higham_correlation.to_numpy(dtype=float),
    q1_labels,
    top_n=5,
)

display(q1_higham_correlation)
display(pd.DataFrame({"Eigenvalue": q1_higham_eigenvalues}))
display(q1_largest_movements)

,x1,x2,x3,x4,x5
x1,1.000000,0.349452,0.425942,0.270782,0.694572
x2,0.349452,1.000000,0.260599,0.307523,0.739337
x3,0.425942,0.260599,1.000000,0.492500,0.662558
x4,0.270782,0.307523,0.492500,1.000000,0.737677
x5,0.694572,0.739337,0.662558,0.737677,1.000000


,Eigenvalue
0,-0.000000
1,0.450082
2,0.703476
3,0.806708
4,3.039734


,Pair,Before,After,Change,Absolute Change
0,x2-x5,0.787538,0.739337,-0.048201,0.048201
1,x4-x5,0.782632,0.737677,-0.044955,0.044955
2,x1-x5,0.732272,0.694572,-0.037700,0.037700
3,x3-x5,0.684257,0.662558,-0.021699,0.021699
4,x2-x4,0.287709,0.307523,0.019814,0.019814


The Higham-repaired correlation matrix is

$$
\begin{bmatrix}
1.000000 & 0.349452 & 0.425942 & 0.270782 & 0.694572\\
0.349452 & 1.000000 & 0.260599 & 0.307523 & 0.739337\\
0.425942 & 0.260599 & 1.000000 & 0.492500 & 0.662558\\
0.270782 & 0.307523 & 0.492500 & 1.000000 & 0.737677\\
0.694572 & 0.739337 & 0.662558 & 0.737677 & 1.000000
\end{bmatrix}.
$$

Its eigenvalues are approximately -0.0000, 0.450082, 0.703476, 0.806708, and 3.039734. The tiny negative first value showing -0.000 is due to numerical rounding, so the nearest matrix obtained is PSD to numerical tolerance.

The five largest correlation changes are:

| Pair | Before | After | Change |
|---|---:|---:|---:|
| x2–x5 | 0.787538 | 0.739337 | **-0.048201** |
| x4–x5 | 0.782632 | 0.737677 | **-0.044955** |
| x1–x5 | 0.732272 | 0.694572 | **-0.037700** |
| x3–x5 | 0.684257 | 0.662558 | **-0.021699** |
| x2–x4 | 0.287709 | 0.307523 | **+0.019814** |

### Question 1(e): Combine the full-sample marginal variances with the repaired correlation matrix to obtain a covariance matrix.

Apply the formula $\Sigma=D R D$, where $R$ is the repaired correlation matrix and $D$ contains the marginal sample standard deviations.

In [6]:
# FUNCTION CALL: repaired correlation plus full-sample marginal volatilities.
q1_covariance = pd.DataFrame(
    covariance_from_correlation(
        q1_higham_correlation.to_numpy(dtype=float), q1_standard_deviations
    ),
    index=q1_labels,
    columns=q1_labels,
)
display(q1_covariance)

,x1,x2,x3,x4,x5
x1,0.000406,0.000154,0.000106,0.000158,0.000231
x2,0.000154,0.000480,0.000070,0.000195,0.000268
x3,0.000106,0.000070,0.000151,0.000176,0.000135
x4,0.000158,0.000195,0.000176,0.000842,0.000354
x5,0.000231,0.000268,0.000135,0.000354,0.000273


The covariance matrix, in squared-return units, is

$$
\begin{bmatrix}
0.000406074 & 0.000154291 & 0.000105511 & 0.000158311 & 0.000231324\\
0.000154291 & 0.000480066 & 0.000070189 & 0.000195487 & 0.000267728\\
0.000105511 & 0.000070189 & 0.000151110 & 0.000175647 & 0.000134608\\
0.000158311 & 0.000195487 & 0.000175647 & 0.000841739 & 0.000353716\\
0.000231324 & 0.000267728 & 0.000134608 & 0.000353716 & 0.000273150
\end{bmatrix}.
$$

# Question 2 — VaR and ES for a Single Stock

Using `problem2.csv`, calculate arithmetic returns for XYZ and SPY, fit the market model

$$r_{XYZ}=\alpha+\beta r_{SPY}+\epsilon,$$

assume that SPY returns and $\epsilon$ are Normal and independent, and report XYZ VaR and ES at 95% and 99% confidence as positive percentages.

### Question 2(a): Calculate the 95% and 99% VaR for the 100-share XYZ position, expressed as percentages.
Use arithmetic returns from `calculate_returns()` and estimate $\alpha$ and $\beta$ with `fit_market_model()`. Fit Normal distributions to SPY returns and the regression residuals. Under independence, XYZ is Normal with

$$\mu_{XYZ}=\alpha+\beta\mu_{SPY}+\mu_\epsilon,$$

$$\sigma_{XYZ}=\sqrt{\beta^2\sigma_{SPY}^2+\sigma_\epsilon^2}.$$

Finally, `value_at_risk()` returns the positive magnitude of the lower-tail loss. It shows the threshold number at 99% or 95%, instead of showing the expection of the whole tail. The number of shares changes dollar VaR but not percentage VaR. 

In [7]:
# INPUT AND FUNCTION CALL: arithmetic returns.
q2_prices = pd.read_csv(DATA / "problem2.csv")
q2_returns = calculate_returns(
    q2_prices, method="DISCRETE", date_column="Date", demean=False
)

# FUNCTION CALL: market-model regression.
q2_market_model = fit_market_model(q2_returns["XYZ"], q2_returns["SPY"])
q2_spy_fit = fit_normal(q2_returns["SPY"])
q2_residual_fit = fit_normal(q2_market_model.residuals)

q2_xyz_mean = (
    q2_market_model.alpha
    + q2_market_model.beta * q2_spy_fit.mu
    + q2_residual_fit.mu
)
q2_xyz_standard_deviation = np.sqrt(
    (q2_market_model.beta * q2_spy_fit.sigma) ** 2
    + q2_residual_fit.sigma**2
)
q2_xyz_distribution = stats.norm(
    loc=q2_xyz_mean, scale=q2_xyz_standard_deviation
)

q2_risk_table = pd.DataFrame(
    [
        {
            "Confidence": 1 - tail_alpha,
            "Tail Alpha": tail_alpha,
            "VaR": value_at_risk(q2_xyz_distribution, alpha=tail_alpha),
            "ES": expected_shortfall(q2_xyz_distribution, alpha=tail_alpha),
        }
        for tail_alpha in (0.05, 0.01)
    ]
)
q2_risk_table["VaR Percent"] = 100 * q2_risk_table["VaR"]
q2_risk_table["ES Percent"] = 100 * q2_risk_table["ES"]

display(pd.DataFrame([{
    "Regression alpha": q2_market_model.alpha,
    "Regression beta": q2_market_model.beta,
    "SPY mean": q2_spy_fit.mu,
    "SPY standard deviation": q2_spy_fit.sigma,
    "Residual standard deviation": q2_residual_fit.sigma,
    "Modeled XYZ mean": q2_xyz_mean,
    "Modeled XYZ standard deviation": q2_xyz_standard_deviation,
}]))
display(q2_risk_table[["Confidence", "Tail Alpha", "VaR", "VaR Percent"]])

,Regression alpha,Regression beta,SPY mean,SPY standard deviation,Residual standard deviation,Modeled XYZ mean,Modeled XYZ standard deviation
0,-0.000304,1.193311,0.000255,0.010972,0.014590,0.000000,0.019604


,Confidence,Tail Alpha,VaR,VaR Percent
0,0.950000,0.050000,0.032245,3.224494
1,0.990000,0.010000,0.045605,4.560464


The fitted model has $\alpha=-0.000304022$, $\beta=1.193311$, and modeled daily XYZ std 0.019603534. The modeled daily mean is effectively zero.
At 95% confidence ($\alpha=5\%$), VaR is a positive loss of 3.2245%.
At 99% confidence ($\alpha=1\%$), VaR is a positive loss of 4.5605%.

### Question 2(b): Calculate the 95% and 99% Expected Shortfall for XYZ, expressed as percentages, and explain the method.
Use  `expected_shortfall()` function on the same fitted Normal XYZ distribution. ES is the expected value returned on a lower tail, thus it is able to show the performance of the whole 1% or 5% lower tail.

In [8]:
display(q2_risk_table[["Confidence", "Tail Alpha", "ES", "ES Percent"]])

,Confidence,Tail Alpha,ES,ES Percent
0,0.950000,0.050000,0.040436,4.043646
1,0.990000,0.010000,0.052248,5.224762




- At 95% confidence ($\alpha=5\%$), ES is a positive loss of 4.0436%.
- At 99% confidence ($\alpha=1\%$), ES is a positive loss of 5.2248%.

ES is larger than VaR at both of the confidence level because ES averages losses beyond the threshold calculated by VaR. VAR only reported the number at the threshold.

# Question 3 — Copula VaR and ES for a Two-Book Portfolio

### Question 3(a): Fit Normal, Student-t, and Normal inverse Gaussian distributions to every stock's arithmetic returns; report the fitted parameters and identify the best model for each stock.

To fit all three distributions, use function `fit_normal()`, `fit_student_t()`, and `fit_nig_mle()` for each stock. Then compare the models with `aicc()` which is the AICcand choose the distribution with the smallest AICc. AICc is combined of two factors, which is likelihood and additional parameters.The formula shows number of parameters to discourage complexity, and minus it by likelihood. The likelihood is encouraged to be bigger. Thus lower AICc is more preferred.

In [9]:
# INPUT AND FUNCTION CALL: arithmetic stock returns.
q3_prices = pd.read_csv(DATA / "problem3.csv")
q3_holdings = pd.read_csv(DATA / "problem3_holdings.csv")
q3_assets = [column for column in q3_prices.columns if column != "Date"]
q3_returns = calculate_returns(
    q3_prices, method="DISCRETE", date_column="Date", demean=False
)[q3_assets]

q3_margin_rows = []
q3_selected_margins = {}

for asset in q3_assets:
    values = q3_returns[asset].to_numpy(dtype=float)

    # FUNCTION CALLS: three candidate marginal distributions.
    normal_fit = fit_normal(values)
    student_fit = fit_student_t(values)
    nig_fit = fit_nig_mle(values)
    candidates = {
        "Normal": normal_fit,
        "Student-t": student_fit,
        "Normal inverse Gaussian": nig_fit,
    }
    model_aicc = {
        model_name: aicc(model_fit, values)
        for model_name, model_fit in candidates.items()
    }
    selected_name = min(model_aicc, key=model_aicc.get)
    q3_selected_margins[asset] = candidates[selected_name]

    q3_margin_rows.extend([
        {
            "Stock": asset,
            "Distribution": "Normal",
            "mu": normal_fit.mu,
            "sigma_or_scale": normal_fit.sigma,
            "nu": np.nan,
            "alpha": np.nan,
            "beta": np.nan,
            "delta": np.nan,
            "AICc": model_aicc["Normal"],
            "Selected": selected_name == "Normal",
        },
        {
            "Stock": asset,
            "Distribution": "Student-t",
            "mu": student_fit.mu,
            "sigma_or_scale": student_fit.sigma,
            "nu": student_fit.nu,
            "alpha": np.nan,
            "beta": np.nan,
            "delta": np.nan,
            "AICc": model_aicc["Student-t"],
            "Selected": selected_name == "Student-t",
        },
        {
            "Stock": asset,
            "Distribution": "Normal inverse Gaussian",
            "mu": nig_fit.mu,
            "sigma_or_scale": np.nan,
            "nu": np.nan,
            "alpha": nig_fit.alpha,
            "beta": nig_fit.beta,
            "delta": nig_fit.delta,
            "AICc": model_aicc["Normal inverse Gaussian"],
            "Selected": selected_name == "Normal inverse Gaussian",
        },
    ])

q3_margin_table = pd.DataFrame(q3_margin_rows)
display(q3_margin_table)
display(q3_margin_table.loc[q3_margin_table["Selected"], [
    "Stock", "Distribution", "AICc"
]].reset_index(drop=True))

,Stock,Distribution,mu,sigma_or_scale,nu,alpha,beta,delta,AICc,Selected
0,S1,Normal,0.000904,0.011826,NaN,NaN,NaN,NaN,"-6,033.993247",True
1,S1,Student-t,0.000904,0.011820,"355,368,542.589316",NaN,NaN,NaN,"-6,031.981687",False
2,S1,Normal inverse Gaussian,0.076557,NaN,NaN,"2,086.485600",-588.204192,0.257473,"-6,030.215060",False
3,S2,Normal,0.000800,0.015348,NaN,NaN,NaN,NaN,"-5,512.705402",False
4,S2,Student-t,0.000581,0.011673,4.857877,NaN,NaN,NaN,"-5,627.359694",True
5,S2,Normal inverse Gaussian,-0.000305,NaN,NaN,73.206598,4.863823,0.016593,"-5,622.063046",False
6,S3,Normal,0.000424,0.018898,NaN,NaN,NaN,NaN,"-5,096.512392",False
7,S3,Student-t,0.002291,0.013657,4.160482,NaN,NaN,NaN,"-5,238.463137",False
8,S3,Normal inverse Gaussian,0.010474,NaN,NaN,79.836689,-36.344624,0.019657,"-5,281.640956",True
9,S4,Normal,0.000780,0.009933,NaN,NaN,NaN,NaN,"-6,382.973454",True


,Stock,Distribution,AICc
0,S1,Normal,"-6,033.993247"
1,S2,Student-t,"-5,627.359694"
2,S3,Normal inverse Gaussian,"-5,281.640956"
3,S4,Normal,"-6,382.973454"
4,S5,Student-t,"-5,150.869475"
5,S6,Normal inverse Gaussian,"-5,737.395453"
6,S7,Normal,"-5,743.693780"
7,S8,Student-t,"-5,493.287554"
8,S9,Normal inverse Gaussian,"-5,065.509637"
9,S10,Student-t,"-5,974.857010"


The fitted parameters and AICc values are below. For Normal, parameters are $(\mu,\sigma)$; for Student-t, $(\mu,\text{scale},\nu)$; and for NIG, $(\mu,\alpha,\beta,\delta)$. The lowest AICc for each stock is marked **Best**.

| Stock | Distribution | Fitted parameters | AICc | Choice |
|---|---|---|---:|---|
| S1 | Normal | μ=0.000904, σ=0.011826 | -6033.993 | **Best** |
| S1 | Student-t | μ=0.000904, scale=0.011820, ν=355368542.589 | -6031.982 |  |
| S1 | NIG | μ=0.076557, α=2086.485600, β=-588.204192, δ=0.257473 | -6030.215 |  |
| S2 | Normal | μ=0.000800, σ=0.015348 | -5512.705 |  |
| S2 | Student-t | μ=0.000581, scale=0.011673, ν=4.857877 | -5627.360 | **Best** |
| S2 | NIG | μ=-0.000305, α=73.206598, β=4.863823, δ=0.016593 | -5622.063 |  |
| S3 | Normal | μ=0.000424, σ=0.018898 | -5096.512 |  |
| S3 | Student-t | μ=0.002291, scale=0.013657, ν=4.160482 | -5238.463 |  |
| S3 | NIG | μ=0.010474, α=79.836689, β=-36.344624, δ=0.019657 | -5281.641 | **Best** |
| S4 | Normal | μ=0.000780, σ=0.009933 | -6382.973 | **Best** |
| S4 | Student-t | μ=0.000780, scale=0.009928, ν=339036291.501 | -6380.962 |  |
| S4 | NIG | μ=0.089549, α=3368.189374, β=-984.957946, δ=0.290287 | -6379.120 |  |
| S5 | Normal | μ=0.000709, σ=0.018951 | -5090.895 |  |
| S5 | Student-t | μ=0.000833, scale=0.015577, ν=6.289248 | -5150.869 | **Best** |
| S5 | NIG | μ=0.001497, α=73.576752, β=-2.232562, δ=0.025958 | -5146.210 |  |
| S6 | Normal | μ=0.000805, σ=0.015085 | -5547.203 |  |
| S6 | Student-t | μ=0.002445, scale=0.010655, ν=3.823756 | -5688.740 |  |
| S6 | NIG | μ=0.008272, α=91.682513, β=-41.686207, δ=0.014628 | -5737.395 | **Best** |
| S7 | Normal | μ=0.000693, σ=0.013674 | -5743.694 | **Best** |
| S7 | Student-t | μ=0.000692, scale=0.013668, ν=319103.362911 | -5741.682 |  |
| S7 | NIG | μ=0.077482, α=1635.963518, β=-443.712607, δ=0.272611 | -5739.888 |  |
| S8 | Normal | μ=0.000374, σ=0.017112 | -5295.106 |  |
| S8 | Student-t | μ=0.000654, scale=0.011424, ν=3.483289 | -5493.288 | **Best** |
| S8 | NIG | μ=0.001183, α=50.622840, β=-2.886825, δ=0.014161 | -5487.094 |  |
| S9 | Normal | μ=0.001746, σ=0.020647 | -4919.506 |  |
| S9 | Student-t | μ=0.003383, scale=0.015310, ν=4.385214 | -5036.762 |  |
| S9 | NIG | μ=0.010128, α=63.159483, β=-22.926741, δ=0.021515 | -5065.510 | **Best** |
| S10 | Normal | μ=0.000165, σ=0.013061 | -5835.324 |  |
| S10 | Student-t | μ=0.000405, scale=0.009427, ν=4.121270 | -5974.857 | **Best** |
| S10 | NIG | μ=0.001049, α=76.685731, β=-5.368403, δ=0.012595 | -5970.702 |  |

The selected margins are therefore: Normal for S1, S4, and S7; Student-t for S2, S5, S8, and S10; and NIG for S3, S6, and S9.

### Question 3(b):  Fit Gaussian and Student-t copulas using the selected marginal distributions, determine which copula fits better, and justify the choice.
Transform each return series through its selected fitted CDF with `uniforms_from_fitted_margins()`. Then call `fit_copula_models()` and compare log likelihood, AICc, and BIC. Smaller AICc and BIC indicate the preferred model after penalizing complexity.

In [10]:
# FUNCTION CALLS: probability integral transforms and two copula fits.
q3_uniforms = uniforms_from_fitted_margins(
    q3_returns, q3_selected_margins
)
q3_copula_table, q3_copulas = fit_copula_models(q3_uniforms)
q3_selected_copula = q3_copula_table.loc[
    q3_copula_table["AICc"].idxmin(), "Copula"
]

display(q3_copula_table)
print("Selected copula:", q3_selected_copula)

Selected copula: Gaussian


,Copula,Log Likelihood,nu,AICc,BIC
0,Gaussian,"2,110.022340",NaN,"-4,125.705057","-3,909.195692"
1,Student-t,"2,108.163552",100.000000,"-4,119.789854","-3,898.570362"


From the data,
| Copula | Log likelihood | ν | AICc | BIC |
|---|---:|---:|---:|---:|
| Gaussian | **2110.022340** | — | **-4125.705057** | **-3909.195692** |
| Student-t | 2108.163552 | 100.000000 | -4119.789854 | -3898.570362 |

we can see that the Gaussian copula fits better because it has both the lower AICc and the lower BIC. The fitted Student-t copula reaches $\nu=100$, which is already close to a Gaussian copula, and its extra degrees-of-freedom parameter does not improve the fit as it adds to complexity.

### Question 3(c): Using the selected copula, simulate at least 5,000 scenarios and calculate positive 95% dollar VaR and ES for every stock, each book, and the total portfolio. Present the results and discuss them.

use the selected Gaussian copula with 100,000 simulations and random seed 545. 
Positions are valued as final price times shares held. Normal and Student-t margins are inverted directly. For the three selected NIG margins, SciPy's exact NIG random generator is used, and the draws are matched to the copula uniforms by rank. The empirical inverse-CDF implementation  is able to preserve the fitted NIG marginal distribution and the copula ranks. Dollar P&L equals simulated arithmetic return times current position value. 
Functions `value_at_risk()` and `expected_shortfall()` return positive loss magnitudes at $\alpha=5\%$, or 95% confidence.

In [11]:
# FUNCTION CALL: simulate only the selected Gaussian copula.
q3_simulated_uniforms = simulate_gaussian_copula(
    q3_copulas["gaussian_correlation"],
    n_samples=N_SIMULATIONS,
    seed=RANDOM_SEED,
)

# Transform copula uniforms through the selected margins.
q3_simulated_returns = np.empty_like(q3_simulated_uniforms)
for index, asset in enumerate(q3_assets):
    fitted_margin = q3_selected_margins[asset]
    frozen_margin = scipy_distribution(fitted_margin)

    if isinstance(fitted_margin, NormalInverseGaussianFit):
        # Efficient empirical inverse CDF for the fitted NIG margin.
        generator = np.random.default_rng(RANDOM_SEED + index + 1)
        marginal_draws = frozen_margin.rvs(
            size=N_SIMULATIONS, random_state=generator
        )
        copula_order = np.argsort(q3_simulated_uniforms[:, index])
        q3_simulated_returns[copula_order, index] = np.sort(marginal_draws)
    else:
        q3_simulated_returns[:, index] = frozen_margin.ppf(
            q3_simulated_uniforms[:, index]
        )

# Current values and simulated dollar P&L.
q3_holding_by_stock = q3_holdings.set_index("Stock").reindex(q3_assets)
q3_last_prices = q3_prices[q3_assets].iloc[-1].to_numpy(dtype=float)
q3_shares = q3_holding_by_stock["Holding"].to_numpy(dtype=float)
q3_current_values = q3_last_prices * q3_shares
q3_pnl_by_stock = q3_simulated_returns * q3_current_values

# FUNCTION CALLS: positive loss VaR and ES for stocks, books, and total.
q3_risk_rows = []
for index, asset in enumerate(q3_assets):
    stock_pnl = q3_pnl_by_stock[:, index]
    q3_risk_rows.append({
        "Position": asset,
        "Level": "Stock",
        "Current Value ($)": q3_current_values[index],
        "VaR 95% ($)": value_at_risk(stock_pnl, alpha=0.05),
        "ES 95% ($)": expected_shortfall(stock_pnl, alpha=0.05),
    })

for book in q3_holdings["Book"].drop_duplicates():
    member_indices = [
        index
        for index, asset in enumerate(q3_assets)
        if q3_holding_by_stock.loc[asset, "Book"] == book
    ]
    book_pnl = q3_pnl_by_stock[:, member_indices].sum(axis=1)
    q3_risk_rows.append({
        "Position": f"Book {book}",
        "Level": "Book",
        "Current Value ($)": q3_current_values[member_indices].sum(),
        "VaR 95% ($)": value_at_risk(book_pnl, alpha=0.05),
        "ES 95% ($)": expected_shortfall(book_pnl, alpha=0.05),
    })

q3_total_pnl = q3_pnl_by_stock.sum(axis=1)
q3_risk_rows.append({
    "Position": "Total Portfolio",
    "Level": "Portfolio",
    "Current Value ($)": q3_current_values.sum(),
    "VaR 95% ($)": value_at_risk(q3_total_pnl, alpha=0.05),
    "ES 95% ($)": expected_shortfall(q3_total_pnl, alpha=0.05),
})

q3_risk_table = pd.DataFrame(q3_risk_rows)
q3_risk_table["VaR Percent"] = (
    q3_risk_table["VaR 95% ($)"] / q3_risk_table["Current Value ($)"]
)
q3_risk_table["ES Percent"] = (
    q3_risk_table["ES 95% ($)"] / q3_risk_table["Current Value ($)"]
)

display(q3_risk_table)
print(f"Simulation size: {N_SIMULATIONS:,}; random seed: {RANDOM_SEED}")

Simulation size: 100,000; random seed: 545


,Position,Level,Current Value ($),VaR 95% ($),ES 95% ($),VaR Percent,ES Percent
0,S1,Stock,"48,329.478358",895.480937,"1,133.563218",0.018529,0.023455
1,S2,Stock,"46,697.425991","1,084.445607","1,576.331301",0.023223,0.033756
2,S3,Stock,"24,267.276940",801.659019,"1,196.879578",0.033035,0.049321
3,S4,Stock,"43,596.810952",679.706329,860.974353,0.015591,0.019749
4,S5,Stock,"37,727.450468","1,097.900478","1,550.187778",0.029101,0.041089
5,S6,Stock,"46,326.642687","1,219.679537","1,870.753950",0.026328,0.040382
6,S7,Stock,"40,144.306138",876.964999,"1,106.878005",0.021845,0.027572
7,S8,Stock,"26,045.880281",643.735746,"1,017.883077",0.024715,0.039080
8,S9,Stock,"98,248.712042","3,379.125462","5,150.160154",0.034394,0.052420
9,S10,Stock,"24,512.380167",479.649286,720.018652,0.019568,0.029374


All VaR and ES figures below are positive dollar loss magnitudes at 95% confidence ($\alpha=5\%$), based on 100,000 simulations and seed 545.

| Position | Current value | VaR 95% | ES 95% | VaR / value | ES / value |
|---|---:|---:|---:|---:|---:|
| S1 | $48,329.48 | **+$895.48** | **+$1,133.56** | 1.8529% | 2.3455% |
| S2 | $46,697.43 | **+$1,084.45** | **+$1,576.33** | 2.3223% | 3.3756% |
| S3 | $24,267.28 | **+$801.66** | **+$1,196.88** | 3.3035% | 4.9321% |
| S4 | $43,596.81 | **+$679.71** | **+$860.97** | 1.5591% | 1.9749% |
| S5 | $37,727.45 | **+$1,097.90** | **+$1,550.19** | 2.9101% | 4.1089% |
| S6 | $46,326.64 | **+$1,219.68** | **+$1,870.75** | 2.6328% | 4.0382% |
| S7 | $40,144.31 | **+$876.96** | **+$1,106.88** | 2.1845% | 2.7572% |
| S8 | $26,045.88 | **+$643.74** | **+$1,017.88** | 2.4715% | 3.9080% |
| S9 | $98,248.71 | **+$3,379.13** | **+$5,150.16** | 3.4394% | 5.2420% |
| S10 | $24,512.38 | **+$479.65** | **+$720.02** | 1.9568% | 2.9374% |
| **Book A** | **$200,618.44** | **+$3,611.70** | **+$4,910.16** | **1.8003%** | **2.4475%** |
| **Book B** | **$235,277.92** | **+$5,402.75** | **+$7,883.42** | **2.2963%** | **3.3507%** |
| **Total Portfolio** | **$435,896.36** | **+$7,689.68** | **+$10,720.21** | **1.7641%** | **2.4593%** |


Discussion: S9 has the largest stand-alone dollar risk because it combines the largest current position value with a heavy, asymmetric NIG margin. Book B has greater VaR and ES than Book A.

The total portfolio VaR (USD 7,689.68) is lower than the combined VaR of Books A and B (USD 9,014.45). Similarly, the total portfolio ES (USD 10,720.21) is lower than the combined ES of Books A and B (USD 12,793.58). Therefore, diversification reduces aggregate risk even though the fitted cross-stock correlations are positive.